In [161]:
from math import *
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
from scipy.integrate import quad
from ODE_solvers import RungeKutta4
from prettytable import PrettyTable as ptable
import time
from scipy.integrate import trapezoid
from scipy.optimize import bisect
from scipy import optimize
start_time = time.time()
import importlib

In [162]:
import lib_Star
# Принудительная перезагрузка модуля
importlib.reload(lib_Star)
# Теперь импортируем класс
from lib_Star import Star, Star_2

import lib_GDF
# Принудительная перезагрузка модуля
importlib.reload(lib_GDF)
# Теперь импортируем класс
from lib_GDF import tay_gdf, epsilon_gdf, pi_gdf, q_gdf, z_gdf

import lib_zeta
# Принудительная перезагрузка модуля
importlib.reload(lib_zeta)
# Теперь импортируем класс
from lib_zeta import zeta

In [163]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 16
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:bold'

pd.set_option('display.max_colwidth', None)  # Снять ограничение на ширину колонок
pd.set_option('display.width', None)  # Снять ограничение на ширину вывода
pd.set_option('display.max_columns', None)  # Показать все колонки


In [164]:
Test = 1 # 0 - единичный расчет; 1 - метод Монте-Карло

In [165]:
p_nom = 20e6
bar_e_0 = 0.36
#name = 'С-5' # 0.057
#name = 'С-8' # 0.08
#name = 'С-13' # 0.122
#name = '9М27Ф' # 0.22
#name = 'С-24' # 0.24
#name = '9М55' # 0.3
name = 'Х-15' # 0.455

In [166]:
n = 7
bet = 35.56*pi/180

In [167]:
# Загрузка данных о топливе
#propellant = pd.read_csv('propellant test.csv').set_index('propellant').T
propellant = pd.read_csv('propellant 13ПБГГ_87ПХА.csv').set_index('propellant').T
class Propellant:
    def __init__(self, df):
        for key in df.index:
            setattr(self, key, df[key])

pr = Propellant(propellant.iloc[0])

In [168]:
import pandas as pd
import numpy as np

# Читаем Excel
df = pd.read_excel("database-1-2026-09-24.xlsx", sheet_name="Лист1")

key = (name, bar_e_0, p_nom)

row = df[
    (df["Name"] == key[0]) &
    (df["bar_e_0"] == key[1]) &
    (df["p_nom"] == key[2])
].iloc[0]

omega_ign = row["omega_ign"]
e_0_ign   = row["e_0_ign"]
omega_t   = row["omega_t"]
print(f"omega_t = {omega_t}")
print(f"omega_ign = {omega_ign}")
print(f"e_0_ign = {e_0_ign}")

if Test == 0:
    e_0   = row["e_0"]
    D_a   = row["D_a"]
    D_ks  = row["D_ks"]
    D_kr  = row["D_kr"]
    r     = row["r"]
    l_zar = row["l_zar"]
    print(f"D_ks = {D_ks}")
    print(f"D_kr = {D_kr}")
    print(f"D_a = {D_a}")
    print(f"e_0 = {e_0}")
    print(f"r = {r}")
    print(f"l_zar = {l_zar}")
else:
    if Test == 1:
        df_geom = pd.read_excel(
            f"{name}-{bar_e_0}-{p_nom/1e6:.0f}-Разброс геометрии.xlsx",
            sheet_name="Sheet1"
        )

        # Случайная выборка N_MC комбинаций из всех
        N_MC = 750
        np.random.seed(41)  # для воспроизводимости
        df_mc = df_geom.sample(n=N_MC, random_state=42).reset_index(drop=True)

        # === ГЛАВНОЕ ИСПРАВЛЕНИЕ ===
        # Все значения геометрии — как массивы (для покомпонентных вычислений)
        D_ks  = df_mc["D_ks"].to_numpy()
        e_0   = df_mc["e_0"].to_numpy()
        r     = df_mc["r"].to_numpy()
        l_zar = df_mc["l_zar"].to_numpy()
        D_a   = df_mc["D_a"].to_numpy()
        D_kr  = df_mc["D_kr"].to_numpy()

        # для удобства — case_id
        case_id = df_mc["case_id"].to_numpy()

        print(f"Загружено {N_MC} геометрий (Test=1, Монте-Карло)")
        print(f"D_ks : shape={D_ks.shape}, min={D_ks.min():.6f}, max={D_ks.max():.6f}")
        print(f"e_0  : shape={e_0.shape},  min={e_0.min():.6f},  max={e_0.max():.6f}")
        print(f"r    : shape={r.shape},    min={r.min():.6f},    max={r.max():.6f}")
        print(f"l_zar: shape={l_zar.shape},min={l_zar.min():.6f},max={l_zar.max():.6f}")
        print(f"D_a  : shape={D_a.shape},  min={D_a.min():.6f},  max={D_a.max():.6f}")
        print(f"D_kr : shape={D_kr.shape}, min={D_kr.min():.6f}, max={D_kr.max():.6f}")

omega_t = 403.8
omega_ign = 2.7
e_0_ign = 0.0006
Загружено 750 геометрий (Test=1, Монте-Карло)
D_ks : shape=(750,), min=0.435737, max=0.437900
e_0  : shape=(750,),  min=0.077900,  max=0.079300
r    : shape=(750,),    min=0.012250,    max=0.013150
l_zar: shape=(750,),min=2.001800,max=2.004925
D_a  : shape=(750,),  min=0.370555,  max=0.370644
D_kr : shape=(750,), min=0.083973, max=0.084027


In [169]:
        # Список объектов Star_2 — по одному на каждую геометрию
star_2 = [
            Star_2(
                D=D_ks[i], e_0=e_0[i], omega=omega_t,
                rho_t=pr.rho, n=n, bet=bet, r=r[i]
            )
            for i in range(N_MC)
        ]

 # Все параметры — в один DataFrame
params_2_list = [s.get_params_2() for s in star_2]
df_params = pd.DataFrame(params_2_list)
df_params.insert(0, "case_id", case_id)

        # Массивы промежуточных величин
e_1     = df_params["e_1"].to_numpy()
e_max   = df_params["e_max"].to_numpy()
s_sr   = df_params["s_sr"].to_numpy()
s_sr0   = df_params["s_sr0"].to_numpy()
kappa_0 = df_params["kappa_0"].to_numpy()

        #print(f"\nПосчитано {len(stars_2)} объектов Star_2")
        #print(f"e_max   : min={e_max_arr.min():.6f},   max={e_max_arr.max():.6f}")
        #print(f"kappa_0 : min={kappa_0_arr.min():.4f}, max={kappa_0_arr.max():.4f}")

In [170]:
def S_g(e):
    """Возвращает массив S_g(e) для всех 500 геометрий."""
    return np.array([s.S_g(e) for s in star_2])

def kappa(e):
    """Возвращает массив kappa(e) для всех 500 геометрий."""
    return np.array([s.kappa(e) for s in star_2])

In [171]:
if name == 'test':
    igniter = pd.read_csv('igniter test.csv').set_index('igniter').T
else:
    igniter = pd.read_csv('igniter.csv').set_index('igniter').T
class Igniter:
    def __init__(self,df):
        for key in df.index:
            setattr(self,key,df[key])
            
ign = Igniter(igniter.iloc[0])  

In [172]:
ds = pd.read_csv('dataset_star.csv').set_index('key').to_dict()

ds_values = ds['value']
ds_converted = {}
for key, value in ds_values.items():
    try:
        # Пробуем преобразовать в float
        ds_converted[key] = float(value)
    except (ValueError, TypeError):
        ds_converted[key] = value

[ c_p_init, R_g_init, dt_1, dt_2, t_1] = list(ds_converted.values())


In [173]:
T_0_norm = 293.15
p_h =0.1e6
G_ras_history = { '+20': []}
time_history = { '+20': []}
# начальные температуры
T_0 = {
       '+20': T_0_norm
}
# цвета для температур
cvet = {
        '+20': 'limegreen'
}


kappa_por = 100


In [174]:
F_ks = np.pi/4 * D_ks**2
d_eff = 4 * l_zar / kappa_0  # эффективный диаметр канала
d_kan = d_eff
F_kan = S_g(0) / kappa_0
F_kr = pi * D_kr**2 / 4
F_a = pi * D_a**2 / 4
q = F_kr / F_a

In [175]:
R_ign_eff = ign.R_g * (1 - ign.z)

In [176]:
kappa_por = 100
d_ign = 5 *e_0_ign
c_ign = 0.75 * e_0_ign

G_ras_history = {'+20': []}
time_history = {'+20': []}
P_history = {'+20': []}

# начальные температуры
T_0 = {
       '+20': T_0_norm,
       }
# цвета для температур
cvet = {
        '+20': 'limegreen',
        }

In [177]:
#### для двояковыпуклой таблетки (одного зерна)

# радиус r таблетки и высота h и радиус R сферических сегментов таблетки
r_ign = d_ign/2
h_sph_ign = e_0_ign - c_ign/2
R_sph_ign = (r_ign**2 + h_sph_ign**2)/2/h_sph_ign

# ф-ия площади пов-ти горения от толщины сгоревшего свода для ПЕРВОЙ стадии
def S_1(e):
    h_1 = R_sph_ign - e - sqrt((R_sph_ign - e)**2 - (r_ign - e)**2)
    c_1 = 2*(e_0_ign - e - h_1)
    return 4*pi*(R_sph_ign - e)*h_1 + 2*pi*(r_ign - e)*c_1

# условие конца ПЕРВОЙ стадии с_1(e_ign_1) = 0  ---->  e_ign_1 = ...
e_ign_1 = (R_sph_ign**2 - r_ign**2 - (R_sph_ign - e_0_ign)**2)/2/(R_sph_ign - r_ign)

# ф-ия площади пов-ти горения от толщины сгоревшего свода для ВТОРОЙ стадии
def S_2(e):
    return 4*pi*(R_sph_ign - e)*(e_0_ign - e)

# объём ОДНОЙ таблетки (объём единичного зерна воспламенителя)
V_ign_1 = pi*r_ign**2*c_ign + 2*pi*h_sph_ign**2*(R_sph_ign - h_sph_ign/3)

# масса единичного зерна воспламенителя
omega_ign_1 = ign.rho*V_ign_1

# закон изменения площади пов-ти горения единичного зерна воспламенителя
def S_ign_1(e):
    if e <= e_ign_1:
        return S_1(e)
    else:
        return S_2(e)

# суммарная площадь пов-ти горения воспламенителя
def S_g_ign(e):
    return omega_ign/omega_ign_1*S_ign_1(e)

In [178]:
R_eff = pr.R_g*(1 - pr.z)
R_ign_eff = ign.R_g*(1 - ign.z)

# Постоянная расхода
A_n = sqrt(pr.n_1 * (2 / (pr.n_1 + 1))**((pr.n_1 + 1) / (pr.n_1 - 1)))

In [179]:
import time
from scipy.integrate import trapezoid

start_time = time.time()

# Массивы для накопления результатов
results = {
    'case_id': [],
    'D_ks': [], 'e_0': [], 'r': [], 'l_zar': [], 'D_a': [], 'D_kr': [],
    'I_p': [], 'I_sp': [],
    't_osn': [], 't_end': [],
    'status': []  # ok / error
}

# Количество успешных расчётов
success_count = 0
error_count = 0

for idx, row in df_mc.iterrows():
    try:
        # Извлекаем параметры
        D_ks_i  = row['D_ks']
        e_0_i   = row['e_0']
        r_i     = row['r']
        l_zar_i = row['l_zar']
        D_a_i   = row['D_a']
        D_kr_i  = row['D_kr']
        
        # Пересоздаём Star_2 для новой геометрии
        star_2_i = Star_2(
            D=D_ks_i,
            e_0=e_0_i,
            omega=omega_t,
            rho_t=pr.rho,
            n=n,
            bet=bet,
            r=r_i
        )
        star_params_2_i = star_2_i.get_params_2()
        e_max_i    = star_params_2_i['e_max']
        kappa_0_i  = star_params_2_i['kappa_0']
        
        # Функции площади и каппы для этой геометрии
        def S_g_i(e, _s=star_2_i):
            return _s.S_g(e)
        def kappa_i(e, _s=star_2_i):
            return _s.kappa(e)
        
        # Пересчитываем геометрию камеры
        F_ks_i  = np.pi/4 * D_ks_i**2
        F_kr_i  = np.pi * D_kr_i**2 / 4
        F_a_i   = np.pi * D_a_i**2 / 4
        q_a_i   = F_kr_i / F_a_i
        
        # Пересчёт lambda_a (зависит от q_a)
        def f_i(x, _q=q_a_i):
            return _q - q_gdf(x, pr.n_a_fr)
        lambda_a_i = bisect(f_i, 2, 3, xtol=1e-12, maxiter=100)
        
        # Начальный свободный объём
        W_0_i = np.pi/12*D_ks_i**3 + np.pi/4*D_ks_i**2*l_zar_i - omega_t/pr.rho
        
        # Переопределяем систему для данной геометрии
        def system_i(t, x, T_0):
            rho = x[0]/(x[4]*x[1])
            k = x[3]/(x[3] - x[4])
            u = pr.u_1*(x[0]/pr.p_ref)**pr.nu*np.exp(pr.dlnu_dT*(T_0 - pr.T_ref))
            u_ign = ign.u_1*(x[0]/ign.p_ref)**ign.nu*np.exp(ign.dlnu_dT*(T_0 - pr.T_ref))
            
            if kappa_i(x[6]) >= kappa_por:
                phi_kapa = 1 + 0.003*(kappa_i(x[6]) - kappa_por)
            else:
                phi_kapa = 1
            
            G_T = S_g_i(x[6])*u*pr.rho*phi_kapa \
                  *np.heaviside(e_max_i - x[6], 0.5) \
                  *(1 - np.heaviside(pr.T_ign - T_0 - sqrt(x[7]), 0.5))
            G_ign = S_g_ign(x[5])*u_ign*ign.rho \
                    *np.heaviside(e_0_ign - x[5], 0.5)
            
            # Геометрия канала
            d_eff_i = 4*l_zar_i/kappa_0_i
            d_kan_i = d_eff_i
            F_kan_i = S_g_i(0)/kappa_0_i
            
            Re = G_ign*d_kan_i/(F_kan_i*ign.mu_g)
            Pr_ = ign.c_pg*ign.mu_g/ign.lambda_g
            Nu = 0.023*Re**0.8*Pr_**0.4
            q_zar = Nu*ign.lambda_g/d_kan_i*(x[1] - T_0 - sqrt(x[7]))
            
            X = G_ign*ign.c_p*(ign.T_p - x[1]) + G_T*pr.c_p*(pr.T_p - x[1]) \
                - q_zar*S_g_i(0)*np.heaviside(pr.T_ign - T_0 - sqrt(x[7]), 0.5)
            Y = G_ign*R_ign_eff*x[1] + G_T*R_eff*x[1] \
                - x[0]/rho*G_i(x) - x[0]*(G_ign/ign.rho + G_T/pr.rho)
            
            dx_dt = np.zeros(8)
            dx_dt[0] = (k-1)/x[2]*(X + k/(k-1)*Y)
            dx_dt[1] = (k-1)/(rho*x[2]*x[4])*(X + Y)
            dx_dt[2] = G_ign/ign.rho + G_T/pr.rho
            dx_dt[3] = 1/(rho*x[2])*(G_ign*(ign.c_p - x[3]) + G_T*(pr.c_p - x[3]))
            dx_dt[4] = 1/(rho*x[2])*(G_ign*(R_ign_eff - x[4]) + G_T*(R_eff - x[4]))
            dx_dt[5] = u_ign*np.heaviside(e_0_ign - x[5], 0.5)
            dx_dt[6] = u*phi_kapa*np.heaviside(e_max_i - x[6], 0.5) \
                       *(1 - np.heaviside(pr.T_ign - T_0 - sqrt(x[7]), 0.5))
            dx_dt[7] = 2*q_zar**2/(pr.c_pr*pr.lambda_pr*pr.rho) \
                       *np.heaviside(pr.T_ign - T_0 - sqrt(x[7]), 0.5)
            return dx_dt
        
        # Функция расхода G для этой геометрии
        def G_i(x, _Fkr=F_kr_i, _la=lambda_a_i):
            p_crit = p_h / pi_gdf(1.0, pr.n_1)
            if x[0] < p_crit:
                ratio = p_h / x[0]
                term = 1.0 - ratio**((pr.n_1 - 1)/pr.n_1)
                return (x[0]*_Fkr/sqrt(x[4]*x[1])) * ratio**(1.0/pr.n_1) \
                       * sqrt(2.0*pr.n_1/(pr.n_1 - 1.0)*term)
            else:
                return A_n * x[0] * _Fkr / sqrt(x[4]*x[1])
            
        # Функция коэффициента тяги (локально для этой геометрии)
        def koff_i(x, n_, z_, _la=lambda_a_i):
            return epsilon_gdf(x, n_) * (x**2 + 1) * zeta(x, n_, z_)

        # Функция тяги для этой геометрии
        def P_i(x, _Fa=F_a_i, _la=lambda_a_i):
            if x[0] * pi_gdf(_la, pr.n_a_fr) >= 0.3 * p_h:
                return x[0] * _Fa * koff_i(_la, pr.n_a_fr, pr.z) - p_h * _Fa
            else:
                return 0.0
        # Функция отчёта
        def report_i(t, x):
            return [kappa_i(x[6]), x[0]/1e6, G_i(x), P_i(x)]
        
        def stop1_i(t, x):
            return t_1 - t
        def stop2_i(t, x):
            return x[0] - 2e5
        
        def sys_norm_i(t, x):
            return system_i(t, x, T_0_norm)
        
        # Начальные условия
        init = [p_h, T_0_norm, W_0_i, c_p_init, R_g_init, 0, 0, 0]
        
        # Решение
        res_1 = RungeKutta4(init, sys_norm_i, 0, dt_1, 100_000, stop1_i, report_i)
        res_2 = RungeKutta4(res_1[-2, 1:9], sys_norm_i, res_1[-2, 0], 
                            dt_2, 100_000, stop2_i, report_i)
        
        # Объединение
        t_c = np.concatenate([res_1[:, 0], res_2[:, 0]])
        v_c = np.concatenate([res_1[:, 1:9], res_2[:, 1:9]], axis=0)
        r_c = np.concatenate([res_1[:, 9:13], res_2[:, 9:13]], axis=0)
        res_all = np.column_stack([t_c, v_c, r_c])
        
        cols = ['t','p','T','W','c_p','R','e_ign','e','eta_T',
                'kapa','p_MPa','G','P']
        df_i = pd.DataFrame(res_all, columns=cols)
        
        # Извлекаем ключевые точки
        t_arr = df_i['t'].values
        p_arr = df_i['p'].values
        
        idx_ign_arr = np.argwhere(df_i['e'].values > 0)
        idx_osn_arr = np.argwhere(df_i['e'].values <= e_0_i)
        idx_max_arr = np.argwhere(p_arr == p_arr.max())
        idx_end_arr = np.argwhere(p_arr <= 2e5)
        
        if (len(idx_ign_arr) == 0 or len(idx_osn_arr) == 0 
                or len(idx_max_arr) == 0 or len(idx_end_arr) == 0):
            raise ValueError("Не найдены ключевые точки")
        
        i_ign = idx_ign_arr[0][0]
        i_osn = idx_osn_arr[-1][0]
        i_max = idx_max_arr[-1][0]
        i_end = idx_end_arr[-1][-1]
        
        # Интегралы
        #I_p = trapezoid(df_i['P'].values[:i_end+1], t_arr[:i_end+1])
        #m_n = trapezoid(df_i['G'].values[:i_end+1], t_arr[:i_end+1])
        I_p = trapezoid(df_i['P'], df_i['t'])
        m_n = trapezoid(df_i['G'], df_i['t'])
        I_sp = I_p / m_n if m_n > 0 else np.nan
        
        # Записываем результат

        results['case_id'].append(row.get('case_id', idx))
        results['D_ks'].append(D_ks_i)
        results['e_0'].append(e_0_i)
        results['r'].append(r_i)
        results['l_zar'].append(l_zar_i)
        results['D_a'].append(D_a_i)
        results['D_kr'].append(D_kr_i)
        results['I_p'].append(I_p)
        results['I_sp'].append(I_sp)
        results['t_osn'].append(t_arr[i_osn])
        results['t_end'].append(t_arr[i_end])
        results['status'].append('ok')
        
        success_count += 1
        
        # Прогресс каждые 50 расчётов
        if (idx + 1) % 50 == 0:
            elapsed = time.time() - start_time
            eta = elapsed / (idx+1) * (N_MC - idx - 1)
            print(f"[{idx+1}/{N_MC}] успешно: {success_count}, "
                  f"ошибок: {error_count}, прошло: {elapsed:.1f}с, "
                  f"осталось: ~{eta:.1f}с")
    
    except Exception as ex:
        error_count += 1
        results['case_id'].append(row.get('case_id', idx))
        for key in ['D_ks','e_0','r','l_zar','D_a','D_kr',
                    'I_p','I_sp','t_osn','t_end']:
            results[key].append(np.nan)
        results['status'].append(f'error: {type(ex).__name__}: {ex}')
        if error_count <= 5:
            print(f"  Ошибка на case_id={row.get('case_id', idx)}: {ex}")

total_time = time.time() - start_time
print(f"\n=== Готово ===")
print(f"Успешных: {success_count}/{N_MC}")
print(f"Ошибок:   {error_count}/{N_MC}")
print(f"Общее время: {total_time:.1f}с ({total_time/60:.1f} мин)")
print(f"Среднее на расчёт: {total_time/N_MC:.3f}с")

[50/750] успешно: 50, ошибок: 0, прошло: 56.4с, осталось: ~789.8с
[100/750] успешно: 100, ошибок: 0, прошло: 112.5с, осталось: ~731.3с
[150/750] успешно: 150, ошибок: 0, прошло: 166.1с, осталось: ~664.6с
[200/750] успешно: 200, ошибок: 0, прошло: 219.7с, осталось: ~604.0с
[250/750] успешно: 250, ошибок: 0, прошло: 273.4с, осталось: ~546.8с
[300/750] успешно: 300, ошибок: 0, прошло: 327.1с, осталось: ~490.7с
[350/750] успешно: 350, ошибок: 0, прошло: 380.7с, осталось: ~435.1с
[400/750] успешно: 400, ошибок: 0, прошло: 434.3с, осталось: ~380.0с
[450/750] успешно: 450, ошибок: 0, прошло: 488.0с, осталось: ~325.3с
[500/750] успешно: 500, ошибок: 0, прошло: 541.6с, осталось: ~270.8с
[550/750] успешно: 550, ошибок: 0, прошло: 594.9с, осталось: ~216.3с
[600/750] успешно: 600, ошибок: 0, прошло: 648.8с, осталось: ~162.2с
[650/750] успешно: 650, ошибок: 0, прошло: 702.5с, осталось: ~108.1с
[700/750] успешно: 700, ошибок: 0, прошло: 756.1с, осталось: ~54.0с
[750/750] успешно: 750, ошибок: 0, про

In [180]:
df_results = pd.DataFrame(results)
df_results.to_excel(f"MC_results_{name}_{bar_e_0}_{p_nom/1e6:.0f}.xlsx", 
                    index=False)
print("Результаты сохранены")

Результаты сохранены
